# 01 — Ingest SportLogiq API → UC Volume

Single, parameterized notebook that pulls **the whole SportLogiq surface area**
into a Volume landing zone. Same code runs as a one-shot demo, a daily
incremental, a single-team scope, or a full multi-season backfill.

## Coverage

| Group | Routes covered | Volume target |
|-------|----------------|---------------|
| **Reference** | `get_competitions`, `get_teams`, `get_team_records`, `get_venues`, `get_players`, `get_metric_topics(scope)` | `reference/{entity}/` |
| **Per game** | `get_game`, `get_game_roster`, `get_game_compiled_events`, `get_game_full_events`, `get_player_shift_events`, `get_game_player_toi`, `get_game_metrics(topic)` | `games/{game_id}/{kind}.json` |
| **Season aggregates** | `get_competition_metrics(scope, topic)` for player / team / goalie / opposingteam | `season_metrics/{scope}/{topic}.json` |
| **Cross-references** | `get_xrefnames`, `get_external_*_references` | `xrefs/{xrefname}/{entity}.json` |
| **Player history** | `get_player_team_history` (sampled — full universe is huge) | `player_history/{player_id}.json` |

## Modes (set via `INGEST_MODE` env var)

| Mode | Game selection | When to use |
|------|----------------|-------------|
| `daily` *(default)* | `last_metrics_full_process_time_from = now − 24h` | Scheduled job; only games whose metrics finished re-processing in the last day |
| `season` | All games where `season = NHL_SEASON` | First-time backfill of one season |
| `date_window` | All games where `NHL_START_DATE <= date <= NHL_END_DATE` | Targeted backfill (e.g. one month) |
| `team` | All games for `NHL_TEAM_ID` in `NHL_SEASON` | One-team demo (~82 games) — fastest |

> All modes are **idempotent** — files already present in the Volume are skipped, so re-running picks up where the last run stopped.

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [ ]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "sportlogiq_nhl")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver:  {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:    {UC_CATALOG}.{GOLD_SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

In [ ]:
def get_secret(env_var: str, scope: str = "sportlogiq", key: str | None = None) -> str:
    """Read a secret from .env first, then fall back to a Databricks secret scope.

    Lets the same notebook run from a laptop (.env) or inside a Databricks
    Git Folder (workspace secret scope `sportlogiq`).
    """
    val = os.getenv(env_var)
    if val:
        return val
    try:
        from pyspark.dbutils import DBUtils  # type: ignore
        dbutils = DBUtils(spark)
        return dbutils.secrets.get(scope=scope, key=key or env_var.lower())
    except Exception as e:
        raise RuntimeError(
            f"Could not resolve {env_var}: not in environment and Databricks "
            f"secret scope '{scope}' lookup failed ({e})."
        )

In [ ]:
from sportlogiq_client import SportLogiqAPI
import datetime as _dt
from concurrent.futures import ThreadPoolExecutor, as_completed

api = SportLogiqAPI(
    username=get_secret("SPORTLOGIQ_USERNAME"),
    password=get_secret("SPORTLOGIQ_PASSWORD"),
)

# Mode + windowing
INGEST_MODE = os.getenv("INGEST_MODE", "daily").strip().lower()
NHL_SEASON  = os.getenv("NHL_SEASON", "20232024").strip()
NHL_TEAM_ID = (os.getenv("NHL_TEAM_ID") or "").strip() or None
NHL_START   = (os.getenv("NHL_START_DATE") or "").strip() or None
NHL_END     = (os.getenv("NHL_END_DATE") or "").strip() or None
COMPETITION_ID = os.getenv("NHL_COMPETITION_ID", "1").strip()  # SportLogiq: 1 = NHL
INGEST_WORKERS = int(os.getenv("INGEST_WORKERS", "6"))

print(f"INGEST_MODE  : {INGEST_MODE}")
print(f"NHL_SEASON   : {NHL_SEASON}")
print(f"NHL_TEAM_ID  : {NHL_TEAM_ID or '(league-wide)'}")
print(f"DATE WINDOW  : {NHL_START or '(start of season)'}  ->  {NHL_END or '(now)'}")
print(f"COMPETITION  : {COMPETITION_ID}")
print(f"WORKERS      : {INGEST_WORKERS}")

In [ ]:
# Make sure the Volume + per-entity directory tree exists
spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")

REFERENCE_DIR    = f"{VOLUME_PATH}/reference"
GAMES_DIR        = f"{VOLUME_PATH}/games"
SEASON_DIR       = f"{VOLUME_PATH}/season_metrics"
XREFS_DIR        = f"{VOLUME_PATH}/xrefs"
HISTORY_DIR      = f"{VOLUME_PATH}/player_history"


def upload(path: str, payload, *, overwrite: bool = True) -> int:
    if isinstance(payload, (dict, list)):
        body = json.dumps(payload).encode("utf-8")
    elif isinstance(payload, str):
        body = payload.encode("utf-8")
    else:
        body = payload
    w.files.upload(path, body, overwrite=overwrite)
    return len(body)


def already_uploaded(path: str) -> bool:
    """Cheap idempotency check — returns True if the file already lives in the Volume."""
    try:
        w.files.get_metadata(path)
        return True
    except Exception:
        return False


print("Volume helpers ready.")

## Step 1 — Reference data (teams, venues, players, competitions, metric topics)

Cheap, low-volume routes pulled fresh on every run — these are the dimension
sources for silver/gold.

In [ ]:
def fetch_reference():
    written = []

    # Competitions list (catalog of leagues SportLogiq has data for)
    written.append(("competitions",      upload(f"{REFERENCE_DIR}/competitions.json",
                                                api.get_competitions().json())))
    # Single competition detail (NHL)
    written.append(("competition_nhl",   upload(f"{REFERENCE_DIR}/competition_{COMPETITION_ID}.json",
                                                api.get_competition(COMPETITION_ID).json())))
    # Teams + records
    written.append(("teams",             upload(f"{REFERENCE_DIR}/teams.json",
                                                api.get_teams().json())))
    written.append(("team_records",      upload(f"{REFERENCE_DIR}/team_records.json",
                                                api.get_team_records().json())))
    # Venues
    written.append(("venues",            upload(f"{REFERENCE_DIR}/venues.json",
                                                api.get_venues().json())))
    # Players (league-wide for the season)
    players_path = f"{REFERENCE_DIR}/players_{NHL_SEASON}.json"
    written.append(("players",           upload(players_path,
                                                api.get_players(season=NHL_SEASON).json())))
    # Metric topics — one file per scope
    for scope in ["player", "team", "goalie", "opposingteam"]:
        path = f"{REFERENCE_DIR}/metric_topics_{scope}.json"
        written.append((f"metric_topics_{scope}",
                        upload(path, api.get_metric_topics(scope).json())))

    print(f"Reference data: {len(written)} files written.")
    for name, size in written:
        print(f"  {name:<30s}  {size/1024:8.1f} KB")
    return written


fetch_reference()

## Step 2 — Pick the games to ingest

Selection is fully driven by `INGEST_MODE`. Each game contributes ~6 per-game
files later, so for a one-shot demo `team` (≈82 games) is recommended.

In [ ]:
def pick_game_ids() -> list[str]:
    if INGEST_MODE == "daily":
        ts = (_dt.datetime.now(_dt.timezone.utc) - _dt.timedelta(days=1))
        last_process = ts.strftime("%Y-%m-%dT%H:%M:%S.") + f"{ts.microsecond // 1000:03d}Z"
        return api.get_game_ids_by_process_time(last_process)

    if INGEST_MODE == "season":
        return [g["id"] for g in api.get_games(season=NHL_SEASON).json().get("games", [])]

    if INGEST_MODE == "team":
        if not NHL_TEAM_ID:
            raise ValueError("INGEST_MODE=team requires NHL_TEAM_ID in .env")
        all_games = api.get_games(season=NHL_SEASON).json().get("games", [])
        return [g["id"] for g in all_games
                if str(g.get("home_team_id")) == NHL_TEAM_ID or str(g.get("away_team_id")) == NHL_TEAM_ID]

    if INGEST_MODE == "date_window":
        if not (NHL_START and NHL_END):
            raise ValueError("INGEST_MODE=date_window requires both NHL_START_DATE and NHL_END_DATE")
        all_games = api.get_games(season=NHL_SEASON).json().get("games", [])
        return [g["id"] for g in all_games if NHL_START <= g.get("date", "") <= NHL_END]

    raise ValueError(f"Unknown INGEST_MODE={INGEST_MODE!r}")


game_ids = sorted(set(pick_game_ids()))
print(f"Selected {len(game_ids):,} games for ingest.")
if game_ids[:5]:
    print(f"  sample ids: {game_ids[:5]}")

## Step 3 — Per-game payloads

For every selected `game_id`, fan out to all the per-game routes in parallel.
Each route writes one JSON file under `games/{game_id}/{kind}.json`. Files
already in the Volume are skipped (idempotency).

The per-topic `get_game_metrics` call is batched: we request every published
metric topic for the `player` scope so silver has the full metric grid.

In [ ]:
PLAYER_TOPIC_IDS = [t["id"] for t in api.get_metric_topics("player").json().get("topics", [])]
print(f"Player metric topics to pull per game: {len(PLAYER_TOPIC_IDS)}")


def ingest_game(game_id: str) -> dict:
    base = f"{GAMES_DIR}/{game_id}"
    summary = {"game_id": game_id, "wrote": [], "skipped": [], "failed": []}

    routes = [
        ("game",            lambda: api.get_game(game_id).json()),
        ("roster",          lambda: api.get_game_roster(game_id).json()),
        ("compiled_events", lambda: api.get_game_compiled_events(game_id).json()),
        ("full_events",     lambda: api.get_game_full_events(game_id).json()),
        ("shift_events",    lambda: api.get_player_shift_events(game_id).json()),
        ("player_toi",      lambda: api.get_game_player_toi(game_id).json()),
    ]
    for name, fetch in routes:
        path = f"{base}/{name}.json"
        if already_uploaded(path):
            summary["skipped"].append(name)
            continue
        try:
            upload(path, fetch())
            summary["wrote"].append(name)
        except Exception as e:
            summary["failed"].append((name, str(e)[:120]))

    # Player metrics, one file per topic
    for topic in PLAYER_TOPIC_IDS:
        path = f"{base}/metrics_player_{topic}.json"
        if already_uploaded(path):
            summary["skipped"].append(f"metrics:{topic}")
            continue
        try:
            payload = api.get_game_metrics(
                game_id=game_id,
                topic_id=topic,
                breakdown=["player", "team", "period", "position"],
            ).json()
            upload(path, payload)
            summary["wrote"].append(f"metrics:{topic}")
        except Exception as e:
            summary["failed"].append((f"metrics:{topic}", str(e)[:120]))

    return summary


# Run the per-game fan-out in parallel.
done, total_wrote, total_skipped, total_failed = 0, 0, 0, 0
with ThreadPoolExecutor(max_workers=INGEST_WORKERS) as exe:
    futures = {exe.submit(ingest_game, gid): gid for gid in game_ids}
    for fut in as_completed(futures):
        gid = futures[fut]
        try:
            s = fut.result()
            total_wrote   += len(s["wrote"])
            total_skipped += len(s["skipped"])
            total_failed  += len(s["failed"])
            if s["failed"]:
                print(f"  game {gid}: {len(s['failed'])} routes failed — {s['failed'][:2]}")
        except Exception as e:
            total_failed += 1
            print(f"  game {gid}: hard failure — {str(e)[:120]}")
        done += 1
        if done % 25 == 0 or done == len(game_ids):
            print(f"  progress: {done}/{len(game_ids)} games  "
                  f"(wrote={total_wrote}, skipped={total_skipped}, failed={total_failed})")

print(f"\nPer-game ingest done: {done} games, {total_wrote} wrote, {total_skipped} skipped, {total_failed} failed.")

## Step 4 — Season-wide metric aggregates

`get_competition_metrics(competition_id, scope, topic)` returns season-to-date
roll-ups. Pull every topic for every scope so silver can build season
leaderboards directly without aggregating events.

In [ ]:
def fetch_season_aggregates():
    rows = []
    for scope in ["player", "team", "goalie", "opposingteam"]:
        topics = api.get_metric_topics(scope).json().get("topics", [])
        for topic in topics:
            path = f"{SEASON_DIR}/{scope}/{topic['id']}.json"
            if already_uploaded(path):
                rows.append((scope, topic["id"], "skipped"))
                continue
            try:
                payload = api.get_competition_metrics(
                    competition_id=COMPETITION_ID,
                    scope=scope,
                    topic_id=topic["id"],
                    season=NHL_SEASON,
                ).json()
                size = upload(path, payload)
                rows.append((scope, topic["id"], f"{size/1024:.1f} KB"))
            except Exception as e:
                rows.append((scope, topic["id"], f"FAILED: {str(e)[:80]}"))
    print(f"Season metric files: {len(rows)}")
    for scope, topic_id, status in rows[:10]:
        print(f"  {scope:<14s} topic={topic_id:<10s}  {status}")
    if len(rows) > 10:
        print(f"  ... ({len(rows) - 10} more)")
    return rows


fetch_season_aggregates()

## Step 5 — External cross-references

`get_xrefnames` returns the list of foreign systems SportLogiq is mapped to
(e.g. NHL official IDs). For each name we pull the games / players / teams /
venues mapping in one shot — these power joins to your existing data.

In [ ]:
def fetch_xrefs():
    xrefs = api.get_xrefnames().json().get("xrefnames") or api.get_xrefnames().json()
    if isinstance(xrefs, dict) and "xrefnames" in xrefs:
        xrefs = xrefs["xrefnames"]
    if not isinstance(xrefs, list):
        print("xrefnames response shape unexpected; storing raw payload")
        upload(f"{XREFS_DIR}/_xrefnames_raw.json", xrefs)
        return

    upload(f"{XREFS_DIR}/_xrefnames_index.json", xrefs)
    for name in xrefs:
        for kind, fetch in [
            ("games",   api.get_external_game_references),
            ("players", api.get_external_player_references),
            ("teams",   api.get_external_team_references),
            ("venues",  api.get_external_venue_references),
        ]:
            path = f"{XREFS_DIR}/{name}/{kind}.json"
            if already_uploaded(path):
                continue
            try:
                upload(path, fetch(name).json())
            except Exception as e:
                print(f"  {name}/{kind}: {str(e)[:120]}")
    print(f"Pulled xrefs for {len(xrefs)} systems.")


fetch_xrefs()

## Step 6 — Optional: per-player history sample

`get_player_team_history` is per-player so the full league universe is large.
We sample only the players who appear in the games we just ingested — for a
single-team demo that's ~30 active skaters.

In [ ]:
def sample_player_history(limit: int = 50):
    players_seen: set[str] = set()
    for gid in game_ids[:50]:  # cap at 50 games of roster scan
        try:
            roster = api.get_game_roster(gid).json()
        except Exception:
            continue
        for crew in roster.get("crew", []):
            pid = crew.get("id")
            if pid:
                players_seen.add(str(pid))
        if len(players_seen) >= limit:
            break

    pulled = 0
    for pid in list(players_seen)[:limit]:
        path = f"{HISTORY_DIR}/{pid}.json"
        if already_uploaded(path):
            continue
        try:
            upload(path, api.get_player_team_history(pid).json())
            pulled += 1
        except Exception as e:
            print(f"  player {pid}: {str(e)[:80]}")
    print(f"Pulled history for {pulled} players (cap={limit}).")


sample_player_history(limit=int(os.getenv("PLAYER_HISTORY_LIMIT", "50")))

## Summary

In [ ]:
def list_volume_counts() -> dict:
    out = {}
    for d in [REFERENCE_DIR, GAMES_DIR, SEASON_DIR, XREFS_DIR, HISTORY_DIR]:
        try:
            count = sum(1 for _ in w.files.list_directory_contents(d))
        except Exception:
            count = 0
        out[d.replace(VOLUME_PATH + "/", "")] = count
    return out


print("=" * 60)
print("INGEST SUMMARY")
print("=" * 60)
for sub, n in list_volume_counts().items():
    print(f"  {sub:<20s}  {n} entries")
print(f"\nVolume root: {VOLUME_PATH}")
print("Next: run 02_bronze_autoloader.")